# n004 — Artifact Analysis

This notebook analyses the **complexity and novelty** of artifacts produced across experiment conditions.

**REQUIRES:** `004_artifact_analysis.py` to be run first.

**Outputs** (saved to `<ROOT>/plots/`):
- `A004_artifact_metrics.pdf` — grid of time-series plots for each complexity metric (max/mean/median/min per timestep) plus a combined score column
- `004_artifact_complexity.pdf` — boxplot of the per-artifact combined complexity score, averaged per run
- `004_novelty_tails.pdf` — fraction of artifacts falling in each novelty range, plotted on a log scale

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from core.utils import ROOT
from core.utils.analysis_utils import get_exp_folders, agents_time
from collections import defaultdict
from analysis_scripts.artifact_complexity import ExperimentArtifacts
import pandas as pd
import pickle as pkl
import io
import contextlib
import json
from tqdm import tqdm
import matplotlib.colors as mcolors
import itertools

import importlib
importlib.reload(importlib.import_module('analysis_scripts.plot_utils'))
from analysis_scripts.plot_utils import color_map, plot_params

plt.rcParams.update(plot_params)

## Configuration

Set `EXP_BASE_NAMES` to the list of experiment base names you want to compare. All matching completed runs under `ROOT/logs/` are collected automatically.

- `PLOT_NAMES` — display-friendly labels (defaults to title-cased experiment name).
- `SHOW_STD` — toggle standard-deviation bands on time-series plots.
- `MODEL` — the model used for novelty annotations.

In [ ]:
EXP_BASE_NAMES = [] # TO DEFINE

# Change this if you want other names in the plot
PLOT_NAMES = {exp_name: exp_name.replace("_", " ").title() for exp_name in EXP_BASE_NAMES}

EXP_PATH = ROOT / "logs"
SHOW_STD = False
MODEL = 'claude-sonnet-4-5-20250929'

experiments = {}
for base_name in EXP_BASE_NAMES:
    experiments[base_name] = get_exp_folders(log_path=EXP_PATH, 
                                             exp_name=base_name, 
                                             only_completed=True)

## Data Loading

For each run, loads via `ExperimentArtifacts` (with stdout suppressed to keep output clean) and then separately reads:
- `novelties_<MODEL>.pkl` — novelty score per artifact
- `artifact_metrics.pkl` — per-metric time-series and per-artifact values

Novelty scores are attached directly to each artifact dict (`artifact['novelty']`, `artifact['category']`). Missing files are reported and skipped; the artifact is still included but gets `novelty = -1`.

A consistency check warns if the total artifact count, max artifact index, or max creation timestep are mismatched across data sources.

In [ ]:
artifacts = defaultdict(list)

f = io.StringIO()
experiment_novelties = defaultdict(list)
experiment_metrics = defaultdict(list)
agents_timesteps = defaultdict(list)
for base_name, exps in experiments.items():
    print(f"Processing experiments for base: {base_name}")
    for exp in tqdm(exps, desc=base_name):
        # print("Loading:", exp)
        exp_path = EXP_PATH / exp
        try:
            with contextlib.redirect_stdout(f):
                art = ExperimentArtifacts(exp_path=exp_path, embedding_dimensions=512, save_path=exp_path / "artifact_analysis")
                art.load(force_recalc=False)
                # art._load_raw_artifacts()
            
            try:
                with open(exp_path / 'artifact_analysis' / f'novelties_{MODEL}.pkl', 'rb') as nov_file:
                    novelties = pkl.load(nov_file)
            except FileNotFoundError:
                print(f"Cannot load novelties from {MODEL} of {exp} ")
                novelties = {}

            try:
                with open(exp_path / 'artifact_analysis' / 'artifact_metrics.pkl', 'rb') as metrics_file:
                    metrics = pkl.load(metrics_file)
            except FileNotFoundError:
                print(f"Cannot load metrics from {MODEL} of {exp} ")
                metrics = {}

            # # Load novelty of specified model
            failed = []

            max_art_idx = np.concatenate(list(art.get_artifact_by_creation(force=True).values())).astype(int).max()
            max_ts = max(list(art.get_artifact_by_creation(force=True).keys()))
            if max_art_idx != len(art.all_artifacts)-1 or max_art_idx != max(novelties.keys()) or max_ts < art.last_ts:
                print(f"ISSUE:  Max artifact idx in exp {exp}: {max_art_idx} - total artifacts: {len(art.all_artifacts)-1} - novelties: {max(novelties.keys())} - max ts creation: {max_ts} - max ts {art.last_ts}")
            try: 
                novelties[max_art_idx]
            except KeyError:
                print(f"  Warning: max artifact {max_art_idx} not in novelties for exp {exp}.")

            for art_id, artifact in art.all_artifacts.items():
                try:
                    nov = novelties[art_id]
                except KeyError:
                    failed.append(art_id)
                    nov = -1
                artifact['novelty'] = nov

            artifacts[base_name].append(art)
            experiment_novelties[base_name].append(novelties)
            experiment_metrics[base_name].append(metrics)
            agents_timesteps[base_name].append(agents_time(worldlog_path=EXP_PATH / exp / 'open_gridworld.log'))

            if len(failed) > 0:
                print(f"Exp: {exp_path} failed:\n {failed}")
        except Exception as e:
            print(f"Failed to load experiment at {exp_path}: {e}")
    print()

## Novelty Quality Check

Flags runs whose novelty data is likely corrupt or incomplete:
- Runs where all novelty values are zero and there are no `-1` sentinels (suggests the file was empty or never written).
- Runs with more than 5 artifacts assigned `novelty = -1` (LLM annotation failed for too many artifacts).

These runs are excluded from `exp_novelties_ok` but are still available in the raw `experiment_novelties` dict for debugging.

In [ ]:
exp_novelties_ok = {}
for exp_name, data in experiment_novelties.items():
    exp_novelties_ok[exp_name] = []
    print(f"Exp: {exp_name}", end = ' ')
    to_redo = [False, False, False, False, False]
    for idx, d in enumerate(data):
        total = np.sum(list(d.values()))
        neg = len(np.where(np.array(list(d.values()))==-1)[0])
        if total == 0 and neg == 0:
            to_redo[idx] = True
        if neg > 5:
            to_redo[idx] = True
            print(f"  Exp {idx+1} has {neg} negative novelties.")
        if not to_redo[idx]:
            exp_novelties_ok[exp_name].append(d)
    print(f"To redo: {np.where(to_redo)[0]+1}")

## Complexity Metrics

Artifact complexity is captured by a set of named scalar metrics stored in `artifact_metrics.pkl`. 

Two helpers are defined:
- **`plot_metric(metric_values, ax, color, label)`** — plots a set of per-run time-series on a shared axis. When `SHOW_STD=True` shows a shaded band and scatter markers at run endpoints; otherwise plots individual run traces at low alpha. A dot always marks the final timestep of the mean curve.
- **`compute_combined_metric(metrics_df, abs_max, abs_min)`** — normalises each metric to [0, 1] using the global min/max across all conditions, then sums them into a single `combined` score column.

The large grid figure (`A004_artifact_metrics.pdf`) has 4 rows (max / mean / median / min across artifacts at each timestep) and one column per metric plus a final **Combined** column.

In [ ]:
all_metrics = []
for exp in artifacts:
    for arts in artifacts[exp]:
        exp_metrics = list(arts.metrics.keys())
        all_metrics.extend(exp_metrics)
all_metrics = list(set(all_metrics))
del all_metrics[all_metrics.index('InverseCompressionRate')]
print("Metrics to analyze:", all_metrics)

In [ ]:
def plot_metric(metric_values, ax, color, label):
    max_len = max(len(a) for a in metric_values)
    cumul_values = np.full((len(metric_values), max_len), np.nan)
    for i, arr in enumerate(metric_values):
        cumul_values[i, :len(arr)] = arr

    mean = np.nanmean(cumul_values, axis=0)
    ax.plot(np.arange(max_len), mean, color=color, label=label)
    
    if SHOW_STD:
        std = np.nanstd(cumul_values, axis=0)
        ax.fill_between(np.arange(len(mean)), mean - std, mean + std, color=color, alpha=0.2)
        len_values = [[len(v), mean[(len(v)-1)]] for v in metric_values]
        ax.scatter(*zip(*len_values), color=color, alpha=0.7, s=20)

    else:
        for exp_line in cumul_values:
            ax.plot(np.arange(len(exp_line)), exp_line, color=color, alpha=0.2)

    for val in metric_values:        
        eoe = len(val)-1
        ax.scatter(eoe, mean[eoe], color=color, alpha=1, s=20)
    return 

def compute_combined_metric(metrics_df, abs_max, abs_min):
    combined = 0
    for metric_name in metrics_df.columns:
        metric_value = metrics_df[metric_name]
        norm_val = (metric_value - abs_min[metric_name]) / (abs_max[metric_name] - abs_min[metric_name]) if abs_max[metric_name] > abs_min[metric_name] else 0
        combined += norm_val

    metrics_df['combined'] = combined
    return combined

In [ ]:
fig, axs = plt.subplots(4, len(all_metrics) + 1, figsize=(5 * (len(all_metrics) + 1), 10), sharex=True)

abs_max = {metric: -np.inf for metric in all_metrics}
abs_min = {metric: np.inf for metric in all_metrics}

exp_metrics = {}

for exp_name, exp_metrics_list in experiment_metrics.items():
    mins = defaultdict(list)
    maxs = defaultdict(list)
    means = defaultdict(list)
    medians = defaultdict(list)
    for metric_idx, metric in enumerate(all_metrics):
        for metrics in exp_metrics_list:
            metric_values = metrics.get(metric, {})
            if metric_values:
                min_values = metric_values['metric_by_ts'].get('min', None)
                max_values = metric_values['metric_by_ts'].get('max', None)
                mean_values = metric_values['metric_by_ts'].get('mean', None)
                median_values = metric_values['metric_by_ts'].get('median', None)
                if min_values is not None:
                    mins[metric].append(min_values)
                if max_values is not None:
                    maxs[metric].append(max_values)
                if mean_values is not None:
                    means[metric].append(mean_values)
                if median_values is not None:
                    medians[metric].append(median_values)

        if abs_max[metric] < max([max(m) for m in maxs[metric]]):
            abs_max[metric] = max([max(m) for m in maxs[metric]])
        if abs_min[metric] > min([min(m) for m in mins[metric]]):
            abs_min[metric] = min([min(m) for m in mins[metric]])

        plot_metric(metric_values=maxs[metric], ax=axs[0, metric_idx], color=color_map[exp_name], label=PLOT_NAMES[exp_name])
        plot_metric(metric_values=means[metric], ax=axs[1, metric_idx], color=color_map[exp_name], label=PLOT_NAMES[exp_name])
        plot_metric(metric_values=medians[metric], ax=axs[2, metric_idx], color=color_map[exp_name], label=PLOT_NAMES[exp_name])
        plot_metric(metric_values=mins[metric], ax=axs[3, metric_idx], color=color_map[exp_name], label=PLOT_NAMES[exp_name])

        axs[0, metric_idx].set_title(f"{metric} - Max")
        axs[0, metric_idx].grid(True, alpha=0.3)
        axs[0, metric_idx].set_ylabel("Score")
        axs[1, metric_idx].set_title(f"{metric} - Mean")
        axs[1, metric_idx].grid(True, alpha=0.3)
        axs[1, metric_idx].set_ylabel("Score")
        axs[2, metric_idx].set_title(f"{metric} - Median")
        axs[2, metric_idx].grid(True, alpha=0.3)
        axs[2, metric_idx].set_ylabel("Score")
        axs[3, metric_idx].set_title(f"{metric} - Min")
        axs[3, metric_idx].grid(True, alpha=0.3)
        axs[3, metric_idx].set_ylabel("Score")
        axs[3, metric_idx].set_xlabel("Timestep")
    exp_metrics[exp_name] = [maxs, means, medians, mins]

# Calc the combined metric
for exp_name in exp_metrics:
    maxs, means, medians, mins = exp_metrics[exp_name]
    max_combined = []
    min_combined = []
    mean_combined = []
    median_combined = []
    for i in range(len(experiments[exp_name])):
        max_combined.append(0)
        min_combined.append(0)
        mean_combined.append(0)
        median_combined.append(0)
        for metric in all_metrics:
            m = np.array(maxs[metric][i])
            max_combined[i] += (m - abs_min[metric]) / (abs_max[metric] - abs_min[metric]) if abs_max[metric] - abs_min[metric] != 0 else 0
            m = np.array(mins[metric][i])
            min_combined[i] += (m - abs_min[metric]) / (abs_max[metric] - abs_min[metric]) if abs_max[metric] - abs_min[metric] != 0 else 0
            m = np.array(means[metric][i])
            mean_combined[i] += (m - abs_min[metric]) / (abs_max[metric] - abs_min[metric]) if abs_max[metric] - abs_min[metric] != 0 else 0
            m = np.array(medians[metric][i])
            median_combined[i] += (m - abs_min[metric]) / (abs_max[metric] - abs_min[metric]) if abs_max[metric] - abs_min[metric] != 0 else 0
    plot_metric(metric_values=max_combined, ax=axs[0, -1], color=color_map[exp_name], label=PLOT_NAMES[exp_name])
    plot_metric(metric_values=mean_combined, ax=axs[1, -1], color=color_map[exp_name], label=PLOT_NAMES[exp_name])
    plot_metric(metric_values=min_combined, ax=axs[3, -1], color=color_map[exp_name], label=PLOT_NAMES[exp_name])
    plot_metric(metric_values=median_combined, ax=axs[2, -1], color=color_map[exp_name], label=PLOT_NAMES[exp_name])
    axs[0, -1].set_title(f"Combined - Max")
    axs[0, -1].grid(True, alpha=0.3)
    axs[1, -1].set_title(f"Combined - Mean")
    axs[1, -1].grid(True, alpha=0.3)
    axs[2, -1].set_title(f"Combined - Median")
    axs[2, -1].grid(True, alpha=0.3)
    axs[3, -1].set_title(f"Combined - Min")
    axs[3, -1].grid(True, alpha=0.3)
    axs[3, -1].set_xlabel("Timestep")
    axs[0, -1].set_ylabel("Score")
    axs[1, -1].set_ylabel("Score")
    axs[2, -1].set_ylabel("Score")
    axs[3, -1].set_ylabel("Score")

handles, labels = axs[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(labels), bbox_to_anchor=(0.5, -0.02))

plt.subplots_adjust(bottom=0.12)
plt.tight_layout()
plt.savefig(ROOT / 'plots' / 'A004_artifact_metrics.pdf', dpi=300, bbox_inches="tight")
plt.show()

## Combined Complexity — Boxplot

Builds a per-artifact DataFrame (`metrics_df`) by flattening `metrics_by_artifact` values for each metric across all runs, then computes the normalised combined score for every artifact. Scores are averaged per run (`seed_id`) to give one data point per run, which is then shown as a boxplot per condition.

Saved as `004_artifact_complexity.pdf`.

In [ ]:
metrics_df = []

for exp_name, exp_metrics_list in experiment_metrics.items():
    for seed_id, metrics in enumerate(exp_metrics_list):
        reformatted_metrics = {}
        for metric_idx, metric in enumerate(all_metrics):
            metric_values = list(metrics.get(metric, {}).get('metrics_by_artifact', {}).values())
            reformatted_metrics[metric] = metric_values
        reformatted_metrics_df = pd.DataFrame(reformatted_metrics)
        reformatted_metrics_df['combined'] = compute_combined_metric(reformatted_metrics_df, abs_max, abs_min)
        reformatted_metrics_df['experiment'] = exp_name
        reformatted_metrics_df['seed_id'] = seed_id
        metrics_df.append(reformatted_metrics_df)

metrics_df = pd.concat(metrics_df, ignore_index=True)
combined_df = metrics_df.groupby(['experiment', 'seed_id']).mean()['combined'].reset_index()

In [ ]:
order = experiments  # or list(experiment_metrics.keys()), but use the same everywhere

data = [combined_df[combined_df["experiment"] == exp]["combined"].values for exp in order]
colors = [color_map[exp] for exp in order]

fig, ax = plt.subplots(figsize=(5, 4))

bp = ax.boxplot(
    data,
    positions=np.arange(len(order)),   # <-- make positions 0..N-1 to match your mean markers
    patch_artist=True,
    showfliers=False,
    showcaps=False,
    whis=1,
    medianprops={"linewidth": 2},
    whiskerprops={"linewidth": 1},
    capprops={"linewidth": 1},
    widths=0.8,
)

# mean markers (now aligned)
for x, vals in enumerate(data):
    mean_val = float(np.mean(vals)) if len(vals) else np.nan
    ax.plot(
        x, mean_val,
        marker="o",
        markersize=6,
        markerfacecolor="white",
        markeredgecolor="black",
        markeredgewidth=1.3,
        zorder=5,
    )

# boxes: translucent face, no edges
for patch, c in zip(bp["boxes"], colors):
    r, g, b = mcolors.to_rgb(c)
    patch.set_facecolor((r, g, b, 0.5))
    patch.set_edgecolor("none")

# medians: full opacity, colored
for med, c in zip(bp["medians"], colors):
    r, g, b = mcolors.to_rgb(c)
    med.set_color((r, g, b, 1.0))
    med.set_linewidth(2)

# whiskers/caps: two per box
rep_colors = list(itertools.chain.from_iterable([[c, c] for c in colors]))
for whisk, c in zip(bp["whiskers"], rep_colors):
    r, g, b = mcolors.to_rgb(c)
    whisk.set_color((r, g, b, 1.0))
for cap, c in zip(bp["caps"], rep_colors):
    r, g, b = mcolors.to_rgb(c)
    cap.set_color((r, g, b, 1.0))

ax.set_xticks(np.arange(len(order)))
plot_names = [PLOT_NAMES[exp].split(" ") for exp in order]
ax.set_xticklabels(["\n".join(name) for name in plot_names], rotation=45, ha="right")

ax.set_ylabel("Combined Complexity Score")
plt.tight_layout()
plt.savefig(ROOT / "plots" / "004_artifact_complexity.pdf", dpi=300)
plt.show()

## Novelty

Novelty scores are assigned by the LLM per artifact (stored in `novelties_<MODEL>.pkl`). This section analyses the **distribution of novelty scores** across conditions by binning artifacts into configurable ranges.

- **`novelty_ranges`** — list of `(lower, upper)` tuples defining the bins. Use `float('inf')` for an open upper bound.
- **`novelty_range_handling`** — bracket convention for interval endpoints: `'(]'` (lower-exclusive, upper-inclusive) or `'[)'` (lower-inclusive, upper-exclusive).

For each run and range the count is normalised two ways:
- `norm_length` — count / experiment length (artifacts per timestep)
- `norm_total` — count / total artifacts (fraction of the artifact population)

The tail plot uses `norm_total` on a log scale to highlight differences in the high-novelty tail.

In [ ]:
novelty_ranges = [(0, 0), (0, 3), (3, 4.2), (4.2, float('inf'))]  # Define novelty ranges

novelty_range_handling = '(]' # Define whether ranges are '()' or '[]' or '(]' or '[)'

In [ ]:
rows = []

for base_name, experiment_data in artifacts.items():
    for run_id, exp in enumerate(experiment_data):

        for r in novelty_ranges:
            lower, upper = r
            count = 0

            for art in exp.all_artifacts.values():
                nov = max(art.get("novelty", 0), 0)
                if lower != upper:
                    if novelty_range_handling == '[)':
                        if lower <= nov < upper:
                            count += 1
                    elif novelty_range_handling == '(]':
                        if lower < nov <= upper:
                            count += 1
                else:
                    if nov == lower:
                        count += 1
                

            normalize_by_length = count / exp.last_ts
            normalize_by_total = count / len(exp.all_artifacts)

            rows.append({
                "experiment": base_name,
                "run": run_id,              # seed index
                "novelty_lower": lower,
                "novelty_upper": upper,
                "novelty_range": r,         # tuple (lower, upper)
                "count": count,
                "norm_length": normalize_by_length,
                "norm_total": normalize_by_total
            })

df = pd.DataFrame(rows)

In [ ]:
df_mean = (
    df
    .groupby(["experiment", "novelty_range"], as_index=False)
    .agg(mean_count=("count", "mean"), mean_norm_length=("norm_length", "mean"), mean_norm_total=("norm_total", "mean"))
)

df_std = (
    df
    .groupby(["experiment", "novelty_range"], as_index=False)
    .agg(std_count=("count", "std"), std_norm_length=("norm_length", "std"), std_norm_total=("norm_total", "std"))
)

import scipy.stats as stats

df_ci = (
    df
    .groupby(["experiment", "novelty_range"], as_index=False)
    .agg(
        se_count=("count", lambda x: stats.sem(x)),
        se_norm_length=("norm_length", lambda x: stats.sem(x)),
        se_norm_total=("norm_total", lambda x: stats.sem(x))
    )
)

## Novelty Tail Plot

Plots the mean fraction of artifacts in each novelty range (`norm_total`) per condition on a **log scale**, with standard error available via `SHOW_STD`.

Saved as `004_novelty_tails.pdf`.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5))

for exp_name in experiment_novelties.keys():
    g = df_mean.groupby("experiment").get_group(exp_name).sort_values(
        by="novelty_range",
        key=lambda x: x.apply(lambda r: r[0])
    )

    ax.plot(
        g["novelty_range"].astype(str),
        g["mean_norm_total"],
        marker="o",
        label=PLOT_NAMES[exp_name],
        color=color_map[exp_name],
        linestyle='-.',
        linewidth=0.5,
    )

    if SHOW_STD:
        ax.fill_between(
            g["novelty_range"].astype(str),
            g["mean_norm_total"] - df_std.loc[
                (df_std["experiment"] == exp_name),
                "std_norm_total"
            ].values,
            g["mean_norm_total"] + df_std.loc[
                (df_std["experiment"] == exp_name),
                "std_norm_total"
            ].values,
            color=color_map[exp_name],
            alpha=0.2
        )

    se_values = df_ci.loc[
        (df_ci["experiment"] == exp_name),
        "se_norm_total"
    ].values

    # For symmetric error bars, just use se_values directly
    # For 95% CI, multiply by 1.96
    yerr_lower = se_values  # distance below mean
    yerr_upper = se_values  # distance above mean

if novelty_range_handling == '(]':
    xlabels = [f"({x}, {y}]" if y != float('inf') else f">{x}" for x, y in novelty_ranges]
elif novelty_range_handling == '[)':
    xlabels = [f"[{x}, {y})" if y != float('inf') else f"=>{x}" for x, y in novelty_ranges]
xlabels[0] = f"= 0"

ax.set_xlabel("Novelty range")
ax.set_ylabel("Fraction of artifacts in novelty range (log scale)")
# ax.set_title("Normalized artifact count by novelty range")
ax.set_xticklabels(xlabels, rotation=45)
ax.legend()
ax.set_yscale("log")

plt.tight_layout()
plt.savefig(ROOT / "plots" / "004_novelty_tails.pdf", dpi=300)
plt.show()